In [1]:
import pandas as pd
import numpy as np

from linearmodels import PanelOLS

In [2]:
data = pd.read_csv('cleaned_school_data.csv')  

In [3]:
print(data.groupby("Year")["mean_scale_score"].agg(["mean", "std", "min", "max"]).round(3))

         mean     std    min    max
Year                               
2018  599.896  11.460  555.0  647.0
2019  599.042  11.667  553.0  646.0
2022  599.976  11.671  554.0  650.0


In [4]:
# adjust % poverty to a percentage scale
data['poverty_percentage'] = (data['% Poverty'] * 100).round(3)

In [5]:
# # preliminary DiD variables
# data['post'] = (data['Year'] == 2022).astype(int)
# data['treated_cont'] = data['post'] * data['poverty_percentage']

### Individual Grade Datasets

In [6]:
# grade splits

data_3 = data[
    (data['Grade'] == '3') & 
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]


data_4 = data[
    (data['Grade'] == '4') &  
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_5 = data[
    (data['Grade'] == '5') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_6 = data[
    (data['Grade'] == '6') &   
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_7 = data[
    (data['Grade'] == '7') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]

data_8 = data[
    (data['Grade'] == '8') &
    (data['Student Category'] == 'All Students') &
    (data['Report Category'] == 'School')
]



### Verifying Parallel Trends Assumption in the Pre Period

#### All Grades

In [7]:
# Time-placebo test: use 2018 and 2019 as "pre" period
# this is a method to test the parallel trends assumption by checking for any pre-existing trends in the outcome variable before the treatment period
pre_data = data[data['Year'].isin([2018,2019])].copy() # new pre data
pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage'] # new placebo interaction term for treated

In [8]:
pre_data = pre_data.set_index(['DBN', 'Year']) # set index for PanelOLS, it requires a multi-index with entity and time dimensions

In [9]:
# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model = PanelOLS(
    dependent=pre_data['mean_scale_score'],
    exog=pre_data[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [10]:
print(f"Placebo test coefficient: {parallel_trends_model.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: -0.0016, p-value: 0.5982


```python
Placebo test coefficient: -0.0009, p-value: 0.8371
``` 

This is great!!!! Near zero trend in pre-period, parallel trends assumption met

#### Same thing for individual grades

In [11]:
pre_data3 = data_3[data_3['Year'].isin([2018,2019])].copy() # new pre data
pre_data3['fake_post'] = (pre_data3['Year'] == 2019).astype(int) # treating 2019 as the "post" period in the placebo test
pre_data3['fake_treated_cont'] = pre_data3['fake_post'] * pre_data3['poverty_percentage'] # new placebo interaction term for treated

pre_data3 = pre_data3.set_index(['DBN', 'Year'])


# testing parallel trends with a placebo DiD model using PanelOLS
parallel_trends_model3 = PanelOLS(
    dependent=pre_data3['mean_scale_score'],
    exog=pre_data3[['fake_treated_cont']],
    entity_effects=True,
    time_effects=True,
    weights = pre_data3['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

print(f"Placebo test coefficient: {parallel_trends_model3.params['fake_treated_cont']:.4f}, p-value: {parallel_trends_model3.pvalues['fake_treated_cont']:.4f}")

Placebo test coefficient: 0.0230, p-value: 0.0182


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


In [12]:
results = {}

for i in range(3, 9):
    data = globals()[f"data_{i}"]
    
    pre_data = data[data['Year'].isin([2018, 2019])].copy()
    pre_data['fake_post'] = (pre_data['Year'] == 2019).astype(int)
    pre_data['fake_treated_cont'] = pre_data['fake_post'] * pre_data['poverty_percentage']
    
    pre_data = pre_data.set_index(['DBN', 'Year'])
    
    model = PanelOLS(
        dependent=pre_data['mean_scale_score'],
        exog=pre_data[['fake_treated_cont']],
        entity_effects=True,
        time_effects=True,
        weights=pre_data['number_tested']
    ).fit(cov_type='clustered', cluster_entity=True)
    
    results[i] = model
    
    print(f"Dataset {i} → Coef: {model.params['fake_treated_cont']:.4f}, "
          f"p-value: {model.pvalues['fake_treated_cont']:.4f}")

Dataset 3 → Coef: 0.0230, p-value: 0.0182


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 4 → Coef: 0.0289, p-value: 0.0036
Dataset 5 → Coef: -0.0058, p-value: 0.5272


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


Dataset 6 → Coef: -0.0395, p-value: 0.0011
Dataset 7 → Coef: -0.0396, p-value: 0.0040
Dataset 8 → Coef: -0.0007, p-value: 0.9570


c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)
c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


A very scary output to see when assessing parallel trends. Luckily, I am not the first person to pursue a project using DID and have assumptions fail. The difficulty with this model is the alleged binary of "yes" or "no" when talking about parallel trends violation. It is unrealistic to expect consistency when dealing woth real-world data - the world is imperfect and things tend to fluctuate. <br> 

The question is, how do you move forward from this? Rambachan & Roth (2023) have a different methodological approach involving a sensitivity analysis. Testing for parallel trends is already problematic to begin with, but still may have some insight into pre-trends. This paper suggests that instead of passing or failing the assumption to assess how large of a violation does it need to be to affect results. The researchers developed an HonestDID package, which allows for sensitivity analysis towards the magnitude of the assumption violation. <br>

This is where things get complicated. HonestDID is strictly an R and Stata package. Also, it only works with event studies. This may be a more advanced and insightful model. 

### Event Studies Model

In [13]:
grade_data = pd.read_csv('cleaned_school_data_updated.csv')

In [14]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,level_2_count,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,12.0,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,14.0,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,4.0,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,30.0,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,7.0,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,114.0,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,37.0,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,48.0,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,48.0,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1


In [15]:
grade_data['poverty_percentage'] = (grade_data['% Poverty'] * 100).round(3)

In [16]:
grade_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_2_percentage,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Poverty_Category,poverty_percentage
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2022,All Students,21,594.0,4.0,19.0,...,57.1,4.0,19.0,1.0,4.8,5.0,23.8,0.838,1,83.8
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,4,2022,All Students,30,596.0,6.0,20.0,...,46.7,5.0,16.7,5.0,16.7,10.0,33.3,0.838,1,83.8
2,School,01M015,P.S. 015 ROBERTO CLEMENTE,5,2022,All Students,23,599.0,11.0,47.8,...,17.4,4.0,17.4,4.0,17.4,8.0,34.8,0.838,1,83.8
3,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,...,40.5,13.0,17.6,10.0,13.5,23.0,31.1,0.838,1,83.8
4,School,01M015,P.S. 015 ROBERTO CLEMENTE,3,2019,All Students,27,606.0,1.0,3.7,...,25.9,18.0,66.7,1.0,3.7,19.0,70.4,0.845,1,84.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14462,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2019,All Students,323,593.0,113.0,35.0,...,35.3,73.0,22.6,23.0,7.1,96.0,29.7,0.949,1,94.9
14463,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,6,2018,All Students,111,595.0,37.0,33.3,...,33.3,26.0,23.4,11.0,9.9,37.0,33.3,0.915,1,91.5
14464,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,7,2018,All Students,116,592.0,50.0,43.1,...,41.4,18.0,15.5,0.0,0.0,18.0,15.5,0.915,1,91.5
14465,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,8,2018,All Students,90,591.0,24.0,26.7,...,53.3,16.0,17.8,2.0,2.2,18.0,20.0,0.915,1,91.5


In [17]:
# grade splits

data3 = grade_data[
    (grade_data['Grade'] == '3') & 
    (grade_data['Student Category'] == 'All Students') 
    # (grade_data['Report Category'] == 'School')
]


data4 = grade_data[
    (grade_data['Grade'] == '4') &  
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data5 = grade_data[
    (grade_data['Grade'] == '5') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data6 = grade_data[
    (grade_data['Grade'] == '6') &   
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data7 = grade_data[
    (grade_data['Grade'] == '7') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]

data8 = grade_data[
    (grade_data['Grade'] == '8') &
    (grade_data['Student Category'] == 'All Students') &
    (grade_data['Report Category'] == 'School')
]



In [18]:
data3['poverty_2018'] = data3['poverty_percentage'] * (data3['Year'] == 2018).astype(int)
data3['poverty_2022'] = data3['poverty_percentage'] * (data3['Year'] == 2022).astype(int)

C:\Users\madis\AppData\Local\Temp\ipykernel_50300\737285779.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['poverty_2018'] = data3['poverty_percentage'] * (data3['Year'] == 2018).astype(int)
C:\Users\madis\AppData\Local\Temp\ipykernel_50300\737285779.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data3['poverty_2022'] = data3['poverty_percentage'] * (data3['Year'] == 2022).astype(int)


In [19]:
data3 = data3.set_index(['DBN', 'Year'])

In [20]:
# Verify the index is set correctly before the model
print(data3.index.names)    # should show ['DBN', 'Year']
print(data3.shape)          # should show (n_schools * 3 years, n_cols)

# Also check you have multiple years per school
print(data3.reset_index().groupby("DBN")["Year"].count().value_counts())
# Should show most schools have 3 observations (2018, 2019, 2022)

['DBN', 'Year']
(2324, 21)
Year
3    766
2      9
1      8
Name: count, dtype: int64


In [21]:
model3 = PanelOLS(
    dependent=data3['mean_scale_score'],
    exog=data3[['poverty_2018', 'poverty_2022']],
    entity_effects=True,
    time_effects=True,
    weights=data3['number_tested']
).fit(cov_type='clustered', cluster_entity=True)

print(model3.summary)

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0266
Estimator:                   PanelOLS   R-squared (Between):             -0.0057
No. Observations:                2314   R-squared (Within):              -0.0780
Date:                Thu, Apr 02 2026   R-squared (Overall):             -0.0057
Time:                        19:23:18   Log-likelihood                   -5638.9
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      20.845
Entities:                         782   P-value                           0.0000
Avg Obs:                       2.9591   Distribution:                  F(2,1528)
Min Obs:                       1.0000                                           
Max Obs:                       3.0000   F-statistic (robust):             13.403
                            

### Model 1: Pooled Ordinary Least Squares Model

Pooled OLS treats panel data as if it is one cross-sectional piece of data. Essentially, this is a baseline model where time and group fixed effects are ignored. <br>
https://www.geeksforgeeks.org/artificial-intelligence/pooled-ols-regression/